# NB13 — Fusion v2: Working NB06 VAE + Two-Signal Escalation Rule

**Project:** RanSMAP — real-time 3-way ransomware detection
**Previous:** NB10 (Fusion v1) showed Conti_06 recall 20%→80% but FPR cost: Benign 33%, Legit-Enc 17%.

## Hypothesis (HANDOFF item 13)
Replace the single-signal streaming gate with a **two-signal rule**:
- Escalate Benign → Attack **only when BOTH** streaming P(Ransomware) ≥ 0.70 **AND** VAE MSE > threshold
- Legit-Enc is in-distribution for the VAE → low anomaly → protected even if streaming fires
- Conti_06 is OOD → high anomaly → both signals fire → still caught

## Changes vs NB10
- VAE: re-trained with **proper per-window StandardScaler** (NB06 approach: 96×300, 23-dim)
- Fusion gate: `stream AND vae` (v2) instead of `stream` alone (v1)
- Idle state: only **one** signal fires → keep monitoring


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import json as _json
from pathlib import Path

# ── Paths (sandbox-compatible) ────────────────────────────────────────────────
SANDBOX = '/sessions/gallant-practical-davinci/mnt/Directed Study/Ransomware'
PROJECT = '/home/arshad/Claude/Projects/Directed Study/Ransomware'
FEAT    = Path(SANDBOX) / 'features'

# ── Constants ─────────────────────────────────────────────────────────────────
DEVICE         = torch.device('cpu')
SEED           = 42
STREAM_THRESH  = 0.70
VAE_PERCENTILE = 95
W_STREAM       = 50
STRIDE         = 5
N_WINDOWS      = 300
CLASS_ORDER    = ['Benign', 'Legit-Encryption', 'Ransomware']
BENIGN_IDX, LENC_IDX, RAN_IDX = 0, 1, 2
VARIANT_CLASSES = [f'Conti_0{i}' for i in range(1, 8)]

np.random.seed(SEED); torch.manual_seed(SEED)
print(f"Device: {DEVICE}  |  stream_thresh={STREAM_THRESH}  |  vae_percentile={VAE_PERCENTILE}")

## 1. Load data
Text embeddings (190, 300, 384) for classifier + streaming.
23-dim feature matrices (190, 300, 23) for VAE — loaded from per-execution .npy files.


In [ ]:
meta        = pd.read_csv(FEAT / 'metadata.csv')
split       = np.load(FEAT / 'fewshot_split_cnn.npz', allow_pickle=True)
train_idx   = split['train_idx']       # (96,)
val_idx     = split['val_idx']         # (24,)
test_idx    = split['test_idx']        # (70,)
labels_3way = split['labels_3way']     # (190,) — variants keep fine-grained names

N = len(meta)

# Text embeddings for classifier / streaming
X_emb = np.load(FEAT / 'text_sequences.npy')   # (190, 300, 384)

# 23-dim per-window features for VAE (path-translate project path → sandbox)
print("Loading 23-dim feature matrices ...")
X_feat = np.zeros((N, N_WINDOWS, 23), dtype=np.float32)
for i, row in meta.iterrows():
    fp = row['filepath'].replace(PROJECT, SANDBOX)
    X_feat[i] = np.load(fp)

print(f"X_emb  : {X_emb.shape}")
print(f"X_feat : {X_feat.shape}")
print(f"Split  : train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")
print(f"Val distribution  : {dict(zip(*np.unique(labels_3way[val_idx], return_counts=True)))}")
print(f"Test distribution : {dict(zip(*np.unique(labels_3way[test_idx], return_counts=True)))}")

## 2. Scale features (NB06 approach)
StandardScaler fitted on **per-window 23-dim features** from train execs only
(96 execs × 300 windows = 28,800 windows, 23 features each).
Raw features have extreme range (max ~8.4e22) — scaling is critical.


In [ ]:
X_flat          = X_feat.reshape(-1, 23)          # (190*300, 23)
train_mask_exec = np.zeros(N, dtype=bool)
train_mask_exec[train_idx] = True
train_win_mask  = np.repeat(train_mask_exec, N_WINDOWS)   # (57000,) T/F

scaler   = StandardScaler()
scaler.fit(X_flat[train_win_mask])               # fit on 28,800 train windows
X_scaled = scaler.transform(X_flat).reshape(N, N_WINDOWS, 23).astype(np.float32)

print(f"Raw max  : {np.abs(X_feat).max():.3e}")
print(f"Scaled max: {np.abs(X_scaled).max():.3e}")
print(f"Scaled train mean: {X_scaled[train_idx].mean():.4f}  std: {X_scaled[train_idx].std():.4f}")

## 3. Train VAE (exact NB06 architecture, 150 epochs)
`TimeSeriesVAE`: 6900→512→128→mu/logvar(16)→128→512→6900, BatchNorm+Dropout(0.2).
Beta warm-up 0→1e-3 over 20 epochs. Adam lr=1e-3 wd=1e-5, StepLR(step=50, gamma=0.5).


In [ ]:
class TimeSeriesVAE(nn.Module):
    # Exact NB06 architecture: 6900->512->128->mu/lv(16)->128->512->6900
    def __init__(self, seq_len=300, n_features=23, latent_dim=16):
        super().__init__()
        D = seq_len * n_features   # 6900
        self.seq_len, self.n_features = seq_len, n_features
        self.enc = nn.Sequential(
            nn.Linear(D, 512),   nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2))
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_lv = nn.Linear(128, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 512),        nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, D))

    def encode(self, x):
        h = self.enc(x.view(x.size(0), -1))
        return self.fc_mu(h), self.fc_lv(h)

    def decode(self, z):
        return self.dec(z).view(-1, self.seq_len, self.n_features)

    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu if not self.training else mu + (0.5*lv).exp()*torch.randn_like(mu)
        return self.decode(z), mu, lv

vae = TimeSeriesVAE().to(DEVICE)
opt = optim.Adam(vae.parameters(), lr=1e-3, weight_decay=1e-5)
sch = optim.lr_scheduler.StepLR(opt, step_size=50, gamma=0.5)
Xtr = torch.FloatTensor(X_scaled[train_idx])
dl  = DataLoader(TensorDataset(Xtr), batch_size=16, shuffle=True)

print(f"Model parameters: {sum(p.numel() for p in vae.parameters()):,}")
print("Training 150 epochs ...")
for ep in range(1, 151):
    vae.train()
    beta = 1e-3 * min(1.0, ep / 20)
    for (xb,) in dl:
        xb = xb.to(DEVICE)
        opt.zero_grad()
        r, mu, lv = vae(xb)
        loss = (F.mse_loss(r, xb) +
                beta * (-0.5 * torch.mean(torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1))))
        loss.backward(); opt.step()
    sch.step()
    if ep % 30 == 0:
        print(f"  epoch {ep:3d}/150")
print("Done.")

## 4. VAE anomaly scoring + threshold
Reconstruction MSE using **deterministic mu** (no sampling) — same as NB06.
Threshold = 95th percentile of **training** reconstruction MSE.


In [ ]:
vae.eval()
vae_mse = np.zeros(N, dtype=np.float32)
with torch.no_grad():
    for i in range(N):
        x = torch.FloatTensor(X_scaled[i]).unsqueeze(0).to(DEVICE)
        mu, _ = vae.encode(x)
        recon  = vae.decode(mu)      # deterministic
        vae_mse[i] = F.mse_loss(recon, x, reduction='mean').item()

vae_thresh = float(np.percentile(vae_mse[train_idx], VAE_PERCENTILE))

print(f"Train MSE : mean={vae_mse[train_idx].mean():.4f}  std={vae_mse[train_idx].std():.4f}")
print(f"Val MSE   : mean={vae_mse[val_idx].mean():.4f}  std={vae_mse[val_idx].std():.4f}")
print(f"VAE thresh (95th pct of train): {vae_thresh:.4f}")
print()

# Conti_06 per-execution breakdown
c06_idx = test_idx[labels_3way[test_idx] == 'Conti_06']
print("Conti_06 VAE MSE (sanity — should vary, not all ~0.0001):")
for i in c06_idx:
    flag = '⚠ FIRES' if vae_mse[i] > vae_thresh else '  below'
    print(f"  exec {i}: {vae_mse[i]:.4f}  {flag}")
det_06 = (vae_mse[c06_idx] > vae_thresh).mean()
print(f"Conti_06 VAE detection rate: {det_06:.1%}")
print()

# FPR on val originals at this threshold
val_fpr_vae = (vae_mse[val_idx] > vae_thresh).mean()
print(f"VAE FPR on val originals: {val_fpr_vae:.1%}  (high = overfitting to train set)")
print("Note: small-N VAE (96 train) overfits — val originals score above train 95th pct.")

## 5. Primary classifier
Load `classifier.pt` (NB07b TextEmbeddingClassifier, full 300-window context).
Compute softmax probabilities for all 190 executions.


In [ ]:
class TextEmbeddingClassifier(nn.Module):
    # Exact NB07b architecture -- must match classifier.pt
    def __init__(self, seq_len=300, emb_dim=384, n_classes=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(emb_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU())
        self.classifier   = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, n_classes))
        self.reconstructor = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, emb_dim))
    def forward(self, x):
        features = self.conv(x.permute(0, 2, 1))
        pooled   = features.mean(dim=-1)
        return self.classifier(pooled), self.reconstructor(pooled), pooled

clf = TextEmbeddingClassifier().to(DEVICE)
clf.load_state_dict(torch.load(FEAT / 'classifier.pt',
                                map_location=DEVICE, weights_only=False))
clf.eval()

primary_probs = np.zeros((N, 3), dtype=np.float32)
with torch.no_grad():
    for i in range(N):
        x = torch.FloatTensor(X_emb[i]).unsqueeze(0).to(DEVICE)
        logits, _, _ = clf(x)
        primary_probs[i] = torch.softmax(logits, dim=1).cpu().numpy()[0]

primary_pred = np.array(CLASS_ORDER)[primary_probs.argmax(axis=1)]
val_true = np.array([l if l in CLASS_ORDER else 'Ransomware' for l in labels_3way[val_idx]])
print(f"Primary classifier val accuracy: {(primary_pred[val_idx]==val_true).mean():.3f}")
print()
print("Conti_06 primary probabilities:")
for i in c06_idx:
    p = primary_probs[i]
    print(f"  exec {i}: P(Ben)={p[0]:.3f}  P(Lnc)={p[1]:.3f}  P(Ran)={p[2]:.3f}  → {primary_pred[i]}")

## 6. Streaming inference
Sliding window W=50, stride=5 → 51 positions over 300 windows.
Max P(Ransomware) across all windows = streaming signal.


In [ ]:
eval_idx      = np.concatenate([val_idx, test_idx])
streaming_max = np.zeros(N, dtype=np.float32)

print("Computing streaming max P(Ransomware) for", len(eval_idx), "execs ...")
with torch.no_grad():
    for i in eval_idx:
        best = 0.0
        for start in range(0, N_WINDOWS - W_STREAM + 1, STRIDE):
            x = torch.FloatTensor(X_emb[i, start:start+W_STREAM]).unsqueeze(0).to(DEVICE)
            logits, _, _ = clf(x)
            p = torch.softmax(logits, dim=1).cpu().numpy()[0, RAN_IDX]
            if p > best: best = p
        streaming_max[i] = best

print()
print("Conti_06 streaming max P(Ransomware):")
for i in c06_idx:
    fires = streaming_max[i] >= STREAM_THRESH
    print(f"  exec {i}: {streaming_max[i]:.3f}  {'✓ FIRES' if fires else '✗'}")

stream_06_det = (streaming_max[c06_idx] >= STREAM_THRESH).mean()
print(f"Conti_06 streaming detection rate: {stream_06_det:.1%}  (same as NB10 v1)")

## 7. Two-signal fusion (v2)

**Rules:**
1. Primary leans Legit-Enc → **Benign** (streaming + VAE cannot override — FPR protection)
2. Primary leans Ransomware → **Attack** (trust primary — strong signal)
3. Primary leans Benign → consult both signals:
   - BOTH stream ≥ 0.70 AND vae_mse > thresh → **Attack**
   - ONLY ONE fires → **Idle** (suspicious; keep monitoring)
   - NEITHER fires → **Benign**

**vs NB10 v1:** v1 escalated Benign→Attack on streaming alone. v2 requires both.


In [ ]:
def fuse_v2(p_primary, stream_val, vae_val, vt=vae_thresh, st=STREAM_THRESH):
    p_ben, p_lnc, p_ran = p_primary[BENIGN_IDX], p_primary[LENC_IDX], p_primary[RAN_IDX]
    if p_lnc >= p_ran and p_lnc >= p_ben: return 'Benign'       # Rule 1
    if p_ran >= p_ben and p_ran >= p_lnc: return 'Attack'       # Rule 2
    stream_fires = stream_val >= st
    vae_fires    = vae_val    >  vt
    if stream_fires and vae_fires:    return 'Attack'            # Rule 3a: both fire
    if stream_fires  or  vae_fires:   return 'Idle'             # Rule 3b: one fires
    return 'Benign'                                              # Rule 3c: neither

def fuse_v1(p_primary, stream_val, st=STREAM_THRESH):
    p_ben, p_lnc, p_ran = p_primary[BENIGN_IDX], p_primary[LENC_IDX], p_primary[RAN_IDX]
    if p_lnc >= p_ran and p_lnc >= p_ben: return 'Benign'
    if p_ran >= p_ben and p_ran >= p_lnc: return 'Attack'
    return 'Attack' if stream_val >= st else 'Benign'

def baseline(p_primary):
    return 'Attack' if CLASS_ORDER[p_primary.argmax()] == 'Ransomware' else 'Benign'

preds_v2  = {i: fuse_v2(primary_probs[i], streaming_max[i], vae_mse[i]) for i in eval_idx}
preds_v1  = {i: fuse_v1(primary_probs[i], streaming_max[i])              for i in eval_idx}
preds_clf = {i: baseline(primary_probs[i])                                for i in eval_idx}

print("Fusion outputs (v2 distribution over eval set):")
for v in ['Attack','Benign','Idle']:
    print(f"  {v}: {sum(1 for x in preds_v2.values() if x==v)}")

## 8. Results: variant recall


In [ ]:
print(f"{'='*70}")
print(f"  VARIANT RECALL — test set (70 Conti variants)")
print(f"{'='*70}")
print(f"{'Variant':<14} {'Clf-alone':>10} {'Fusion-v1':>11} {'Fusion-v2':>11}  v2 Idle")
print(f"{'-'*70}")

recall_summary = {}
for vname in VARIANT_CLASSES:
    v_idx  = test_idx[labels_3way[test_idx] == vname]
    n      = len(v_idx)
    nc     = sum(1 for i in v_idx if preds_clf[i]=='Attack')
    nv1    = sum(1 for i in v_idx if preds_v1[i] =='Attack')
    nv2    = sum(1 for i in v_idx if preds_v2[i] =='Attack')
    nidle  = sum(1 for i in v_idx if preds_v2[i] =='Idle')
    mark   = ' ◀' if vname=='Conti_06' else ''
    print(f"{vname+mark:<16} {nc}/{n} ({100*nc/n:.0f}%)   "
          f"{nv1}/{n} ({100*nv1/n:.0f}%)   "
          f"{nv2}/{n} ({100*nv2/n:.0f}%)    {nidle}")
    recall_summary[vname] = {'n':n,'clf':nc,'v1':nv1,'v2':nv2,'idle':nidle}

print(f"{'-'*70}")
for k,lbl in [('clf','Clf-alone'),('v1','Fusion-v1'),('v2','Fusion-v2')]:
    tot = sum(recall_summary[v][k] for v in VARIANT_CLASSES)
    print(f"{'ALL':>16} {lbl:>12}: {tot}/70 ({100*tot/70:.1f}%)")

## 9. Results: FPR on validation set


In [ ]:
print(f"{'='*60}")
print(f"  FALSE-POSITIVE RATE — val set (24 executions)")
print(f"{'='*60}")
print(f"{'Class':<22} {'Clf-alone':>12} {'Fusion-v1':>12} {'Fusion-v2':>12}")
print(f"{'-'*60}")

fpr_summary = {}
for cls in ['Benign', 'Legit-Encryption', 'Ransomware']:
    v_idx = val_idx[labels_3way[val_idx] == cls]
    n     = len(v_idx)
    nc    = sum(1 for i in v_idx if preds_clf[i]=='Attack')
    nv1   = sum(1 for i in v_idx if preds_v1[i] =='Attack')
    nv2   = sum(1 for i in v_idx if preds_v2[i] =='Attack')
    nidle = sum(1 for i in v_idx if preds_v2[i] =='Idle')
    tag   = 'TPR↑' if cls=='Ransomware' else 'FPR↓'
    print(f"{cls:<22} {nc}/{n} ({100*nc/n:.0f}%)  "
          f"{nv1}/{n} ({100*nv1/n:.0f}%)  "
          f"{nv2}/{n} ({100*nv2/n:.0f}%)  [{tag}]  v2 Idle={nidle}")
    fpr_summary[cls] = {'n':n,'clf':nc,'v1':nv1,'v2':nv2,'idle':nidle}

## 10. Conti_06 per-execution breakdown


In [ ]:
print(f"{'='*72}")
print("  CONTI_06 DETAIL — per-execution")
print(f"{'='*72}")
print(f"{'Exec':>6} {'P(Ben)':>7} {'P(Lnc)':>7} {'P(Ran)':>7} {'Stream':>8} "
      f"{'VAE MSE':>9} {'s≥0.7':>6} {'v>thr':>6} {'v1':>8} {'v2':>8}")
print('-'*72)
for i in c06_idx:
    p = primary_probs[i]
    sf = 'Y' if streaming_max[i]>=STREAM_THRESH else 'N'
    vf = 'Y' if vae_mse[i]>vae_thresh else 'N'
    print(f"{i:>6} {p[0]:>7.3f} {p[1]:>7.3f} {p[2]:>7.3f} "
          f"{streaming_max[i]:>8.3f} {vae_mse[i]:>9.4f} "
          f"{sf:>6} {vf:>6} {preds_v1[i]:>8} {preds_v2[i]:>8}")

c06 = recall_summary['Conti_06']
print(f"\nConti_06 summary: clf={c06['clf']}/10  v1={c06['v1']}/10  "
      f"v2={c06['v2']}/10 Attack + {c06['idle']} Idle = "
      f"{c06['v2']+c06['idle']}/10 flagged")
print(f"VAE thresh={vae_thresh:.4f}  (train 95th pct)")

## 11. Analysis: why two-signal didn't reduce FPR

**Root cause: small-N VAE overfitting**

The VAE trains on 96 execs. After 150 epochs it memorises the training set
(train MSE mean=0.16) while held-out originals score ~0.40 — well above the 95th-pct
train threshold of 0.27. So the VAE fires for **most** held-out originals, not just
variants. The two-signal rule then behaves like the single-signal rule for false
positives (since the VAE also fires), but HURTS Conti_06 recall because now the
VAE must agree with streaming (it only fires for 5/10 Conti_06 execs).

**Legit-Enc is theoretically in-distribution**, but small-N overfitting makes val
originals look OOD. Expected fix: Alliance dataset (10× data per class) → better
VAE generalisation → selective anomaly signal.

**Silver lining:** 5 of the 6 Conti_06 misses go to **Idle** (one signal fires)
rather than Benign — the system still "suspects" them and would trigger escalation
protocols.


In [ ]:
# Quantify VAE selectivity: what fraction of val originals fire?
val_vae_fires_by_class = {}
for cls in ['Benign', 'Legit-Encryption', 'Ransomware']:
    v_idx = val_idx[labels_3way[val_idx] == cls]
    n_fire = (vae_mse[v_idx] > vae_thresh).sum()
    val_vae_fires_by_class[cls] = (int(n_fire), int(len(v_idx)))
    print(f"VAE fires on {cls:<20}: {n_fire}/{len(v_idx)} ({100*n_fire/len(v_idx):.0f}%)")

print(f"\nFor two-signal to reduce FPR, VAE must NOT fire on Benign/Legit-Enc.")
print(f"With 96-sample VAE, it fires broadly — fix requires more training data.")
print(f"\nConti_06 both-signals summary:")
for i in c06_idx:
    sf = streaming_max[i]>=STREAM_THRESH
    vf = vae_mse[i]>vae_thresh
    status = 'BOTH→Attack' if (sf and vf) else ('stream-only→Idle' if sf else
             ('vae-only→Idle' if vf else 'neither→Benign'))
    print(f"  exec {i}: stream={streaming_max[i]:.3f} vae={vae_mse[i]:.4f}  {status}")

## 12. Save results


In [ ]:
class NpEncoder(_json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.bool_): return bool(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

out = {
    'config': {
        'stream_thresh': STREAM_THRESH,
        'vae_thresh': vae_thresh,
        'vae_percentile': VAE_PERCENTILE,
        'vae_train_mse_mean': float(vae_mse[train_idx].mean()),
        'vae_train_mse_std': float(vae_mse[train_idx].std()),
        'vae_val_mse_mean': float(vae_mse[val_idx].mean()),
        'vae_val_fpr': float((vae_mse[val_idx]>vae_thresh).mean()),
    },
    'variant_recall': recall_summary,
    'val_fpr': fpr_summary,
    'streaming_max': {int(i): float(streaming_max[i]) for i in eval_idx},
    'vae_mse_eval': {int(i): float(vae_mse[i]) for i in eval_idx},
}

out_path = FEAT / 'fusion_v2_results.json'
with open(out_path, 'w') as f:
    _json.dump(out, f, cls=NpEncoder, indent=2)
print(f"Saved → {out_path}")
print()
print("=== SUMMARY ===")
c06 = recall_summary['Conti_06']
ben = fpr_summary['Benign']
lnc = fpr_summary['Legit-Encryption']
v2t = sum(recall_summary[v]['v2'] for v in VARIANT_CLASSES)
v1t = sum(recall_summary[v]['v1'] for v in VARIANT_CLASSES)
print(f"Overall recall: clf={sum(recall_summary[v]['clf'] for v in VARIANT_CLASSES)}/70"
      f"  v1={v1t}/70 ({100*v1t/70:.1f}%)  v2={v2t}/70 ({100*v2t/70:.1f}%)")
print(f"Conti_06:  v1={c06['v1']}/10  v2={c06['v2']}/10 (+{c06['idle']} Idle)")
print(f"Benign FPR:  v1={ben['v1']}/{ben['n']}  v2={ben['v2']}/{ben['n']} (+{ben['idle']} Idle)")
print(f"Legit-Enc FPR: v1={lnc['v1']}/{lnc['n']}  v2={lnc['v2']}/{lnc['n']} (+{lnc['idle']} Idle)")